# Water Main Breaks – Topeka (Kansas)
**Main break analysis: breaks by material, break type, year and month, mains with repeated breaks and hot spot analysis (Getis-Ord Gi*)**

Data: City of Topeka public ArcGIS REST services (water main break work orders and mains with break counts), loaded in QGIS and saved as GeoPackage `topeka_water.gpkg` (layers `breaks`, `mains`), NAD83 / Kansas North (ftUS), EPSG:3419.

In [ ]:
import os, numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import libpysal, esda
os.makedirs('results', exist_ok=True); os.makedirs('images', exist_ok=True)
FT = 0.3048006096                                   # US survey foot -> m
GPKG = 'topeka_water.gpkg'                          # in Colab: upload the GeoPackage exported from QGIS

## 1 Load and clean

In [ ]:
b0 = gpd.read_file(GPKG, layer='breaks'); m = gpd.read_file(GPKG, layer='mains')
xmin, ymin, xmax, ymax = m.total_bounds
b = b0[b0.geometry.x.between(xmin - 2000, xmax + 2000) & b0.geometry.y.between(ymin - 2000, ymax + 2000)]
b = b[~b.STATUS.isin(['CANCELED'])].copy()
b['YR'] = b.YR.astype('Int64'); b['PIPETYPE'] = b.PIPETYPE.replace('', 'unbekannt').fillna('unbekannt')
b['BREAK_TYPE'] = b.BREAK_TYPE.replace('', 'unbekannt').fillna('unbekannt')
m['len_km'] = m.M_LEN * FT / 1000; m['br_per_km'] = m.Break_Count / m.len_km
print(f'Brüche: {len(b0)} geladen, {len(b)} nach Bereinigung ({len(b0)-len(b)} entfernt: storniert oder Koordinate außerhalb)')
print(f'Leitungen mit Brüchen: {len(m)} Abschnitte, {m.len_km.sum():.1f} km')

## 2 Breaks by material, break type, year and month

In [ ]:
MAT = {'Cast Iron':'Grauguss','Ductile Iron':'Duktiles Gusseisen','Polyvinyl Chloride':'PVC','Other':'Sonstige','unbekannt':'unbekannt'}
TYP = {'Hole':'Loch','Shear':'Querbruch (Shear)','Split':'Längsriss (Split)','unbekannt':'unbekannt'}
mat = b.PIPETYPE.map(MAT).fillna('Sonstige').value_counts().rename('Brüche').to_frame(); mat['Anteil_%'] = (mat.Brüche / len(b) * 100).round(1)
typ = b.BREAK_TYPE.map(TYP).fillna('unbekannt').value_counts().rename('Brüche').to_frame(); typ['Anteil_%'] = (typ.Brüche / len(b) * 100).round(1)
yr = b[b.YR.between(2007, 2025)].groupby('YR').size().rename('Brüche')
bm = b[b.YR.between(2007, 2025) & b.Mon.notna()]
mon = bm.groupby(bm.Mon.astype(int)).size().rename('Brüche')
winter = mon.loc[[12, 1, 2]].sum() / mon.sum() * 100
cost = b.WOCOST[b.WOCOST > 0]
mm = m.groupby('m_material').agg(Abschnitte=('Break_Count','size'), Brüche=('Break_Count','sum'), km=('len_km','sum'))
mm['Brüche_je_km'] = (mm.Brüche / mm.km).round(2); mm = mm.sort_values('Brüche', ascending=False)
print(mat, typ, sep='\n\n')

## 3 Mains with repeated breaks

In [ ]:
top = m[m.len_km >= 0.02].sort_values(['Break_Count','br_per_km'], ascending=False).head(20)[['FACILITYID','m_material','Diameter','Break_Count','len_km','br_per_km']].copy()
top['Laenge_m'] = (top.len_km * 1000).round(0); top['br_per_km'] = top.br_per_km.round(1)   # only sections ≥ 20 m (very short pieces distort breaks per km)
rep = m[m.Break_Count >= 5]
print(f'{len(rep)} Abschnitte mit ≥ 5 Brüchen ({rep.len_km.sum():.1f} km) = {rep.Break_Count.sum()} Brüche ({rep.Break_Count.sum()/m.Break_Count.sum()*100:.1f} %)')
top

## 4 Hot spot analysis – Getis-Ord Gi* on a 250 m hexagon grid (cells along the network, 6 nearest neighbours)

In [ ]:
S = 250 / FT                                          # 250 m in ftUS (flat-to-flat)
r = S / np.sqrt(3)                                    # hexagon radius
bx0, by0, bx1, by1 = b.total_bounds
hexes = []; dx, dy = 1.5 * r, S
for i, x in enumerate(np.arange(bx0 - r, bx1 + 2*r, dx)):
    for y in np.arange(by0 - S, by1 + 2*S, dy):
        yy = y + (S/2 if i % 2 else 0)
        hexes.append(Polygon([(x + r*np.cos(a), yy + r*np.sin(a)) for a in np.radians(np.arange(0, 360, 60))]))
hx = gpd.GeoDataFrame(geometry=hexes, crs=b.crs)
hx['n'] = gpd.sjoin(b[['geometry']], hx, predicate='within').groupby('index_right').size().reindex(hx.index, fill_value=0)
net = m.buffer(S / 2).union_all()                    # study area: cells along the mains network (incl. cells without breaks)
hx = hx[hx.intersects(net) | (hx.n > 0)].reset_index(drop=True)
w = libpysal.weights.KNN.from_dataframe(hx, k=6); w.transform = 'r'   # 6 neighbours = hexagon ring
g = esda.G_Local(hx.n.values.astype(float), w, star=True, permutations=999, seed=42)
hx['z'] = g.Zs; hx['p'] = g.p_sim
def cls(z, p):
    if p <= 0.01: return 'Hot Spot 99 %' if z > 0 else 'Cold Spot 99 %'
    if p <= 0.05: return 'Hot Spot 95 %' if z > 0 else 'Cold Spot 95 %'
    if p <= 0.10: return 'Hot Spot 90 %' if z > 0 else 'Cold Spot 90 %'
    return 'nicht signifikant'
hx['Klasse'] = [cls(z, p) for z, p in zip(hx.z, hx.p)]
hot = hx[hx.Klasse.str.startswith('Hot')]
print(hx.Klasse.value_counts()); print(f'Hot Spots: {len(hot)} Zellen mit {hot.n.sum()} Brüchen ({hot.n.sum()/hx.n.sum()*100:.1f} %)')

## 5 Key figures and results

In [ ]:
yr_rec = yr.loc[2016:2025]
kz = pd.Series({
 'Brüche (bereinigt)': len(b), 'Zeitraum': f'{int(b.YR.min())}–{int(b.YR.max())}',
 'Brüche pro Jahr (2007–2025, Mittel)': round(yr.mean(), 0), 'Jahr mit den meisten Brüchen': f'{int(yr.idxmax())} ({int(yr.max())})',
 'Anteil Grauguss (%)': mat.loc['Grauguss','Anteil_%'], 'Anteil Winter Dez–Feb (%)': round(winter, 1),
 'Kosten gesamt (Mio. USD)': round(cost.sum()/1e6, 1), 'Kosten je Bruch, Median (USD)': round(cost.median(), 0),
 'Leitungen mit Brüchen (km)': round(m.len_km.sum(), 1), 'Abschnitte mit ≥ 5 Brüchen': len(rep),
 'Anteil Brüche auf Abschnitten mit ≥ 5 Brüchen (%)': round(rep.Break_Count.sum()/m.Break_Count.sum()*100, 1),
 'Hot-Spot-Zellen (250 m)': len(hot), 'Anteil Brüche in Hot Spots (%)': round(hot.n.sum()/hx.n.sum()*100, 1),
}, name='Wert', dtype=object); kz.index.name = 'Kennzahl'
with pd.ExcelWriter('results/water_main_breaks_topeka.xlsx') as xw:
    kz.to_frame().to_excel(xw, sheet_name='Kennzahlen'); mat.to_excel(xw, sheet_name='Werkstoff'); typ.to_excel(xw, sheet_name='Bruchart')
    yr.to_frame().to_excel(xw, sheet_name='Jahr'); mon.to_frame().to_excel(xw, sheet_name='Monat'); mm.to_excel(xw, sheet_name='Leitungen_Werkstoff')
    top.drop(columns='len_km').to_excel(xw, sheet_name='Top20_Leitungen', index=False); hx.drop(columns='geometry').to_excel(xw, sheet_name='HotSpots_Zellen', index=False)
kz.to_csv('results/kennzahlen.csv', sep=';', decimal=','); mat.to_csv('results/brueche_je_werkstoff.csv', sep=';', decimal=',')
yr.to_csv('results/brueche_je_jahr.csv', sep=';'); top.drop(columns='len_km').to_csv('results/top20_leitungen.csv', index=False, sep=';', decimal=',')
hx.to_file('results/hotspots_topeka.gpkg', layer='hotspots_250m', driver='GPKG')
print(kz.to_string())

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
COL = {'Hot Spot 99 %':'#B2182B','Hot Spot 95 %':'#EF8A62','Hot Spot 90 %':'#FDDBC7','nicht signifikant':'#EEEEEE',
       'Cold Spot 90 %':'#D1E5F0','Cold Spot 95 %':'#67A9CF','Cold Spot 99 %':'#2166AC'}
fig, ax = plt.subplots(figsize=(10, 9))
for k, c in COL.items():
    s = hx[hx.Klasse == k]
    if len(s): s.plot(ax=ax, color=c, edgecolor='white', linewidth=0.3, label=f'{k} ({len(s)})')
m.plot(ax=ax, color='#555555', linewidth=0.3, alpha=0.6)
ax.legend(loc='lower left', frameon=False, fontsize=8.5); ax.set_axis_off()
ax.set_title(f'Hot Spots der Rohrbrüche (Getis-Ord Gi*, 250-m-Sechsecke) – {len(hot)} Hot-Spot-Zellen mit {hot.n.sum()/hx.n.sum()*100:.0f} % der Brüche'.replace('.', ','), loc='left', fontweight='bold', fontsize=10.5)
plt.tight_layout(); plt.savefig('images/01_Hotspots_Topeka.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(yr.index.astype(int), yr.values, color='#1F3864')
ax.axhline(yr.mean(), color='#C0392B', ls='--', lw=1); ax.text(2007, yr.mean() + 10, f'Mittel {yr.mean():.0f} / Jahr', color='#C0392B', fontsize=9)
ax.set_xticks(range(2007, 2026, 2)); ax.set_ylabel('Brüche'); ax.set_title('Rohrbrüche je Jahr (2007–2025)', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/02_Brueche_je_Jahr.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(8, 4.2))
mn = ['Jan','Feb','Mär','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']
ax.bar(mn, mon.reindex(range(1, 13), fill_value=0).values, color=['#2E86C1' if i in (0, 1, 11) else '#9DB4D3' for i in range(12)])
ax.set_ylabel('Brüche'); ax.set_title(f'Brüche je Monat – {winter:.0f} % im Winter (Dez–Feb)'.replace('.', ','), loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/03_Brueche_je_Monat.png', dpi=150); plt.close()

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
axs[0].barh(mat.index[::-1], mat.Brüche[::-1], color='#1F3864'); axs[0].set_title('Brüche je Werkstoff', loc='left', fontweight='bold')
for i, (v, p) in enumerate(zip(mat.Brüche[::-1], mat['Anteil_%'][::-1])): axs[0].text(v, i, f' {v} ({p:.0f} %)', va='center', fontsize=8.5)
axs[1].barh(typ.index[::-1], typ.Brüche[::-1], color='#C0392B'); axs[1].set_title('Brüche je Bruchart', loc='left', fontweight='bold')
for i, (v, p) in enumerate(zip(typ.Brüche[::-1], typ['Anteil_%'][::-1])): axs[1].text(v, i, f' {v} ({p:.0f} %)', va='center', fontsize=8.5)
for a in axs: a.set_xlim(0, a.get_xlim()[1] * 1.25)
plt.tight_layout(); plt.savefig('images/04_Werkstoff_Bruchart.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(10, 9))
m.plot(ax=ax, color='#9A9A9A', linewidth=0.6)
mc = m[m.Break_Count >= 5].sort_values('Break_Count')
mc.plot(ax=ax, column='Break_Count', cmap='YlOrRd', vmin=5, linewidth=2.8, legend=True, legend_kwds={'shrink': 0.6, 'label': 'Brüche je Abschnitt'})
ax.set_axis_off(); ax.set_title(f'Leitungsabschnitte mit wiederholten Brüchen – {len(rep)} Abschnitte mit ≥ 5 Brüchen (farbig)', loc='left', fontweight='bold', fontsize=10.5)
plt.tight_layout(); plt.savefig('images/05_Wiederholte_Brueche.png', dpi=150); plt.close()
print('charts saved')